In [ ]:
# KD-ReID on Kaggle (plan 8). Settings: Accelerator = GPU T4 x2 (one GPU is used), Internet = On,
# Add-ons > Secrets: GH_TOKEN, input dataset with: Market-1501 zip, CUHK03-NP zip,
# market_vit_clipreid.pth, osnet_x1_0_imagenet.pth. Then "Save Version > Save & Run All".
RUN = "8b_unl_cuhk03"

import os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

IN, TMP = Path("/kaggle/input"), Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/tmp")
RAW, DATA, WORK = TMP / "raw", TMP / "data", Path("/kaggle/working")
REPO = TMP / "KD-ReID"
OUT = WORK / "outputs" / "plan8" / RUN

def sh(cmd):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    return r.returncode

# ===== 0. GPU + code =====
import torch
assert torch.cuda.is_available(), "Settings > Accelerator > GPU T4 x2"
print(torch.__version__, torch.cuda.get_device_name(0), "| python", sys.version.split()[0])
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GH_TOKEN")
if not REPO.exists():
    assert sh(f"git clone -q https://{token}@github.com/cts0424/KD-ReID.git {REPO}") == 0, "clone failed (Internet on? GH_TOKEN secret?)"
sh(f"git -C {REPO} pull -q --ff-only && git -C {REPO} log --oneline -1")
assert sh(f"pip install -q -e {REPO}") == 0
os.chdir(REPO)

# ===== 1. Data: unzip if needed, then copy into the layout the loader expects =====
for z in IN.rglob("*.zip"):
    dst = RAW / z.stem
    if not dst.exists():
        print("unzipping", z.name, flush=True)
        zipfile.ZipFile(z).extractall(dst)

def find_split_dir(*keywords):
    hits = [p.parent for root in (IN, RAW) if root.exists() for p in root.rglob("bounding_box_train")
            if "__macosx" not in str(p).lower() and all(k in str(p).lower() for k in keywords)]
    assert hits, f"no bounding_box_train with {keywords} under /kaggle/input"
    return sorted(hits, key=lambda p: len(str(p)))[0]

def find_file(pattern):
    hits = sorted(IN.rglob(pattern))
    assert hits, f"{pattern} not found in the input dataset"
    return hits[0]

targets = {
    DATA / "Market-1501-v15.09.15": find_split_dir("market"),
    DATA / "cuhk03-np" / "detected": find_split_dir("detected"),
}
for dst, src in targets.items():
    if not dst.exists():
        print(f"copy {src} -> {dst}", flush=True)
        shutil.copytree(src, dst)
clip_ckpt = find_file("market_vit_clipreid*.pth")
osnet_w = find_file("osnet_x1_0*.pth")
print("CLIP-ReID:", clip_ckpt, "| OSNet:", osnet_w)

# ===== 2. Train (resumes from OUT/last.pth if present) =====
cfg = f"configs/plan8/{RUN}.yaml"
assert Path(cfg).exists(), cfg
args = (f"--config {cfg} output_dir={OUT} data.root={DATA} unlabeled.root={DATA} "
        f"teacher.checkpoint={clip_ckpt} model.pretrained_path={osnet_w} data.num_workers=4")
t0 = time.time()
rc = sh(f"python tools/train.py {args}")
print(f"== {RUN}: exit {rc}, {(time.time() - t0) / 3600:.1f} h", flush=True)

# ===== 3. Cross-domain evaluation; everything under /kaggle/working is kept as the output =====
sh(f"python tools/eval_matrix.py --runs {OUT} --datasets market1501:{DATA} cuhk03np:{DATA} "
   f"--out {WORK}/eval_matrix_{RUN}.json")
sh(f"grep -H Final {OUT}/log.txt")
